# 教程 1：观测 —— 不动模型一个参数，看清它在想什么

上一篇里，你已经能用 `make_llm` 把模型跑起来了。但一个只会「跑」的引擎，
和一个你能「看懂」的引擎，是两回事。这一篇我们做一件看似朴素的事：**在
生成过程中，把第 3、5、17 层的 hidden states 拿出来**，过一遍稀疏自编码器
（SAE）的编码器，特征打印到控制台，再落盘成 `sae_features.pkl` 供离线分析。

你可能会问：导出中间激活，用 PyTorch 的 forward hook 不就行了？在 vLLM 这样
的连续批处理引擎里不行——你没有 hook 点，也分不清张量里哪一行属于哪个请求。
PLLM 的做法是让你先**声明**：我要看哪一层、看多宽、多久看一次。声明不是
繁文缛节，而是「只看不改」这个承诺的凭证——正因为引擎知道你的代码只读不写，
它才敢在不打扰 CUDA 流的前提下，把中间张量复制一份交给你。整篇教程下来，
模型输出的每一个 token 都不会因为我们的观测而改变。

我们不一步到位，而是分四步走，每步只引入一个新概念：

1. 导出原始 hidden —— 看清导出的硬边界与张量归谁所有；
2. 观测头需要参数 —— 认识**冻结状态**；
3. 三层一起观测 —— 一个任务读多路输入；
4. 生成的同时打印与落盘 —— 用后台线程并行消费 trace。

## 第一步：导出原始 hidden

先来热身。下面的 workload 只做一件事：每个执行步，把第 3 层 hidden 的前
2 行、前 8 列导出来。写法和教程 0 几乎一样——一个 `OBSERVE` 阶段的任务，
一个 `@pllm.export` 声明导出——区别是这次我们导原始张量而不是行均值，正好
借它看清导出的两条边界：

- `max_rows` / `max_cols` 是**硬边界**：输入帧和输出帧都会被裁到这个尺寸，
  且 `max_rows × max_cols × 8` 不能超过单个事件的 32 KiB 上限。想导更宽，
  就得少导几行；
- 你拿到的是**宿主机持有的 CPU 副本**：复制走门控的 stride 路径，不阻塞
  CUDA 流，之后引擎覆写显存也影响不了你手里这份。

运行后，你会看到按步编号的事件，每条是一个 `(≤2, 8)` 的小张量——两个请求
正好占满两行。数值本身不重要，重要的是：从声明到拿到手，你没有碰模型一行
代码。

In [1]:
from typing import Annotated
import torch
import pllm
from vllm import SamplingParams
from pllm.config import ModelConfig


class RawPeek(pllm.Workload):
    """每个执行步，导出第 3 层 hidden 的前 2 行、前 8 列。"""

    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="raw", every=1, max_rows=2, max_cols=8)
    def observe(self, hidden: Annotated[torch.Tensor, pllm.Hidden(3)]) -> torch.Tensor:
        return hidden.float()   # 输入帧已被硬边界裁好：≤2 行、8 列


llm = pllm.make_llm(workload=RawPeek(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", "bfloat16", 64, 2, 128, 0.2, seed=7))
llm.generate(["The sky is", "Water freezes at"],
    SamplingParams(max_tokens=3, temperature=0.0, ignore_eos=True), use_tqdm=False)
for event in llm.trace_snapshot().events:
    if event.tensor_name == "raw":
        print(f"step {event.step_id}: shape={tuple(event.tensor().shape)}, "
              f"首行={event.tensor()[0, :4].tolist()}")

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 18:40:48,173	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 18:40:48 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:40:49 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:40:51 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:40:51 [model.py:1883] Using max model len 64


... (26 outputs left out)

INFO 09-10 18:40:58 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


INFO 09-10 18:40:58 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.62 s


INFO 09-10 18:41:01 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


step 1: shape=(2, 8), 首行=[-0.0277099609375, -0.015869140625, -0.04541015625, -0.060302734375]
step 2: shape=(2, 8), 首行=[0.1435546875, -0.06591796875, -0.1396484375, -0.044921875]


## 第二步：观测头需要参数 —— 冻结状态

SAE 的编码器是一个矩阵 `W`（字典大小 × hidden 宽度）。问题来了：这个矩阵
不在模型权重里，也不需要训练——它只是观测基础设施，初始化一次，之后只读。
它该放在哪？

PLLM 的答案是**冻结状态**：一个不给优化器的 `pllm.InstanceState`。它的生命
周期一眼就能看完——准入（admission）时运行你给的 `initialize`，这里是正交
初始化，并且用框架发的 generator，所以每次运行的结果可复现；之后任何任务，
包括观测任务，都能用 `pllm.Parameter("encoder", "w3")` 读到它；而它永远不会
被更新。

你可能会好奇：观测任务凭什么能读状态？因为观测的签名约束是「不改变模型
状态」，而**读**不改变任何状态——冻结状态和观测是天生一对。

下面先把单层的 SAE 特征跑通。我们导出的不再是原始 hidden，而是
`ReLU(h @ W^T)`：ReLU 之后零元素的比例就是特征的「稀疏度」，SAE 里最直观的
健康指标。注意 `max_cols=896` 这个值的选择：输入帧要保留整宽（hidden 宽度
896）才能做矩阵乘，而输出只有 64 列，远在边界之内。

In [2]:
DICT = 64  # SAE 字典大小


class SingleLayerSAE(pllm.Workload):
    encoder = pllm.InstanceState(
        {"w3": lambda model: (DICT, model.hidden_size)},
        dtype=torch.float32,
        initialize=lambda tensors, generator: [
            torch.nn.init.orthogonal_(tensor, generator=generator)
            for tensor in tensors.values()],
    )

    # max_cols 同时界定输入帧宽与导出列数：输入要整宽（896），输出只有 64 列
    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="sae3", every=1, max_rows=4, max_cols=896)
    def encode(self, h3: Annotated[torch.Tensor, pllm.Hidden(3)],
               w3: Annotated[torch.Tensor, pllm.Parameter("encoder", "w3")]) -> torch.Tensor:
        return torch.relu(h3.float() @ w3.T)

## 第三步：三层一起观测 —— 多输入任务

真实分析里你很少只看一层。好在任务和输入之间不是一对一：一个任务可以同时
声明多个 `Hidden` 输入和多个 `Parameter` 输入。我们把三层写进同一个
`encode`：每层各算一遍特征，沿列拼接成一个 `(行数, 3 × DICT)` 的张量导出，
宿主机消费时按列块切回三层即可。

你可能会问：为什么是一个任务读三路，而不是写三个任务？因为观测附件的签名
约定就是「一个观测任务」，而多读几路输入没有任何额外代价——那几层 hidden
本来就躺在这一步的计算里。

In [3]:
LAYERS = (3, 5, 17)


class SAEObserver(pllm.Workload):
    """3/5/17 层共享一个冻结编码器状态，逐步导出拼接特征。"""

    encoder = pllm.InstanceState(
        {f"w{layer}": lambda model: (DICT, model.hidden_size) for layer in LAYERS},
        dtype=torch.float32,
        initialize=lambda tensors, generator: [
            torch.nn.init.orthogonal_(tensor, generator=generator)
            for tensor in tensors.values()],
    )

    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="sae", every=1, max_rows=4, max_cols=896)
    def encode(
        self,
        h3: Annotated[torch.Tensor, pllm.Hidden(3)],
        h5: Annotated[torch.Tensor, pllm.Hidden(5)],
        h17: Annotated[torch.Tensor, pllm.Hidden(17)],
        w3: Annotated[torch.Tensor, pllm.Parameter("encoder", "w3")],
        w5: Annotated[torch.Tensor, pllm.Parameter("encoder", "w5")],
        w17: Annotated[torch.Tensor, pllm.Parameter("encoder", "w17")],
    ) -> torch.Tensor:
        features = [torch.relu(h.float() @ w.T)
                    for h, w in ((h3, w3), (h5, w5), (h17, w17))]
        return torch.cat(features, dim=-1)

## 第四步：生成的同时打印与落盘 —— 并行消费 trace

到目前为止，我们都是生成完再回头翻 trace。更实用的姿势是边生成边消费：特征
实时打印，结束后落盘。trace 是一个容量 256、加锁、溢出时丢最旧事件
（oldest-drop）的队列，所以一个后台线程可以在生成进行的同时安全地轮询它，
两不相扰。

游标写法只有一个要点：事件的绝对序号 = `dropped + len(events)`。每次轮询从
`cursor - dropped` 切起，即使有事件被挤出队头也不会错位——前提是轮询比
溢出快，0.01 秒的间隔对本教程的规模绰绰有余。

下面的代码还做了一件工程上该做的事：先 `close()` 掉第一步的引擎、清空显存，
再用同一个 seed 起一个新引擎挂上 `SAEObserver`。运行后你会看到每个执行步
一行日志，三层的稀疏度并排打印；最后写出的 `sae_features.pkl` 是一个
「步数 × 请求数」行、`3 × DICT` 列的特征矩阵，可以直接拿去离线分析。

In [4]:
import gc
import pickle
import threading
import time

llm.close()   # 第一步的引擎退场，显存还给下一位
del llm
gc.collect()
torch.cuda.empty_cache()

llm = pllm.make_llm(workload=SAEObserver(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", "bfloat16", 64, 2, 128, 0.2, seed=7))

drained = []
cursor = 0
done = threading.Event()

def drain():
    global cursor
    while not done.is_set():
        snapshot = llm.trace_snapshot()
        drained.extend(snapshot.events[max(0, cursor - snapshot.dropped):])
        cursor = snapshot.dropped + len(snapshot.events)   # 绝对序号，溢出也不错位
        time.sleep(0.01)

thread = threading.Thread(target=drain)
thread.start()
try:
    outputs = llm.generate(
        ["The sky is", "Water freezes at"],
        SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True),
        use_tqdm=False)
finally:
    done.set()
    thread.join()

# 收尾再取一次，接住生成结束、线程还没来得及看到的事件
snapshot = llm.trace_snapshot()
drained.extend(snapshot.events[max(0, cursor - snapshot.dropped):])

rows = []
for event in drained:
    if event.tensor_name != "sae":
        continue
    features = event.tensor()
    rows.append(features)
    sparsity = [(block == 0).float().mean().item()
                for block in features.split(DICT, dim=-1)]
    print(f"step {event.step_id}: 行数={features.shape[0]}, "
          f"各层稀疏度 {[f'{v:.2f}' for v in sparsity]}")

with open("sae_features.pkl", "wb") as handle:
    pickle.dump({"layers": LAYERS, "dict": DICT, "features": torch.cat(rows)}, handle)
print(f"已写入 sae_features.pkl: {tuple(torch.cat(rows).shape)}")

INFO 09-10 18:41:02 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:41:03 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:41:03 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:41:03 [model.py:1883] Using max model len 64


INFO 09-10 18:41:03 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 18:41:05 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (9 outputs left out)

INFO 09-10 18:41:07 [kv_cache_utils.py:2236] Maximum concurrency for 64 tokens per request: 5073.25x


INFO 09-10 18:41:07 [gpu_worker.py:789] Free memory on device (11.17/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.98 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3832543232` (3.57 GiB) to fit into requested memory, or `--kv-cache-memory=10783356928` (10.04 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.72 GiB.


INFO 09-10 18:41:07 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.89 s


step 1: 行数=2, 各层稀疏度 ['0.48', '0.42', '0.53']
step 2: 行数=2, 各层稀疏度 ['0.47', '0.45', '0.47']
step 3: 行数=2, 各层稀疏度 ['0.45', '0.45', '0.47']
step 4: 行数=2, 各层稀疏度 ['0.41', '0.48', '0.48']
step 5: 行数=2, 各层稀疏度 ['0.45', '0.46', '0.48']
step 6: 行数=2, 各层稀疏度 ['0.49', '0.55', '0.45']
step 7: 行数=2, 各层稀疏度 ['0.55', '0.52', '0.52']
已写入 sae_features.pkl: (14, 192)


## 收尾检查：证明我们什么都没改变

观测的承诺是「只看不改」，口说无凭，我们当场兑现它。两个诊断计数器：
`fallback_count` 必须为零——任何非零都说明执行逃出了签名原语；
`pending_state_releases` 为零则说明状态页都被干净地释放了。最后打印生成的
文本：它和完全不开观测时的输出逐字一致——观测全程没有碰过采样路径。

In [5]:
diagnostics = llm.worker_state().diagnostics
assert diagnostics.fallback_count == 0        # 执行从未逃出签名原语
assert diagnostics.pending_state_releases == 0
llm.close()
for output in outputs:
    print(output.outputs[0].text)   # 与无观测时逐字一致：观测不改变行为

 a beautiful place. It is a place
 0 degrees Celsius and boils at 


## 小结

回看这一篇，你只新学了三个概念，但它们会贯穿后面所有教程：

- **观测的第一原则是不改变模型状态。** 引擎敢把中间张量交给你，靠的就是
  签名里「只读」的承诺；冻结状态（无优化器的 `InstanceState`）天然满足它——
  准入时初始化、处处可读、从不更新；
- **`@export` 的边界是硬的**：`max_rows` / `max_cols` 同时裁输入帧与输出，
  单事件上限 32 KiB；你拿到的总是宿主机持有的 CPU 副本；
- **trace 是有界、加锁、oldest-drop 的队列**：声明式导出往里写，后台线程
  用 `dropped + len(events)` 游标往外读。

下一篇：[教程 2：写自己的训练头](02_custom_head.ipynb)。我们将从「只读」
跨到「会写」——给模型挂一个 Huber 回归头，让参数第一次在生成循环里被真正
更新。你会看到，这一篇的声明式写法原封不动地延伸过去，只是状态不再冻结了。